Build the RISCV target

In [66]:
import tvm
import tvm.relax as relax
from tvm import meta_schedule as ms
from tvm.script import relax as R
from tvm.script import tir as T

target = tvm.target.Target(
    "llvm -mtriple=riscv64-unknown-linux-gnu "
    "-mattr=+64bit,+m,+a,+f,+d,+c,+v "
    "-mcpu=generic-rv64 "
    "-mabi=lp64d "
)

Register intrinsics

In [67]:
from tvm.tir.tensor_intrin.riscv_cpu import (
    register_rvv_add_intrinsics,
    get_max_elems,
)
from tvm.runtime import DataType
from tvm.target.codegen import llvm_get_vector_width

BATCH, CHANNELS, H, W = 14, 23, 67, 99
SHAPE = (BATCH, CHANNELS, H, W)
DTYPE = "float32"

In [68]:
@tvm.script.ir_module
class AddModule:
    @R.function
    def main(x: R.Tensor(SHAPE, DTYPE), y: R.Tensor(SHAPE, DTYPE)):
        with R.dataflow():
            gv = relax.op.add(x, y)
            R.output(gv)
        return gv

In [69]:
with target:
    inventory = register_rvv_add_intrinsics(target)

vlen      = llvm_get_vector_width(target)
dt        = DataType(DTYPE)
max_elems = get_max_elems(vlen, lmul=8, sew=dt.bits)

n_elems   = min(SHAPE[0], max_elems)
kernel_name = f"rvv_add_{n_elems}{dt[0]}{dt.bits}"

print("Using kernel:", kernel_name)
print("Registered:", list(inventory.keys()))

Unexpected exception formatting exception. Falling back to standard exception


error: Function must be decorated
 --> /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/python/tvm/tir/tensor_intrin/riscv_cpu.py:224:5
     |  
 224 |      def rvv_vec_add_impl(
     |      ^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_739930/119817958.py", line 2, in <module>
    inventory = register_rvv_add_intrinsics(target)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "python/tvm_ffi/cython/function.pxi", line 929, in tvm_ffi.core.Function.__call__
  File "python/tvm_ffi/cython/function.pxi", line 1083, in tvm_ffi.core.tvm_ffi_callback
  File "/home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/python/tvm/tir/tensor_intrin/riscv_cpu.py", line 343, in register_rvv_add_intrinsics
    """
    
  File "/home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized

In [62]:
with tvm.transform.PassContext(opt_level=3):
    mod = relax.transform.LegalizeOps()(AddModule)
    mod = tvm.tir.transform.VectorizeLoop()(mod)

#ex = relax.build(mod, target=target)

In [63]:
ex = tvm.compile(mod, target=target)

In [64]:
print(mod["main"].script())

# from tvm.script import relax as R

@R.function
def main(x: R.Tensor((14, 23, 67, 99), dtype="float32"), y: R.Tensor((14, 23, 67, 99), dtype="float32")) -> R.Tensor((14, 23, 67, 99), dtype="float32"):
    with R.dataflow():
        gv = R.call_tir(add, (x, y), out_sinfo=R.Tensor((14, 23, 67, 99), dtype="float32"))
        R.output(gv)
    return gv


In [65]:
import os
OUTPUT_DIR = "output/intrinsics"

ex = tvm.compile(mod, target=target)

so_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.so")
asm_path = os.path.join(OUTPUT_DIR, f"intrinsic_add.asm")

ex.export_library(
    so_path,
    cc="riscv64-linux-gnu-gcc"
)
print(f"    [INFO] Shared lib -> {so_path}")

import subprocess
result = subprocess.run(
    ["llvm-objdump-18", "-d", "--mattr=+v", so_path],
    capture_output=True, text=True, check=True
)
with open(asm_path, "w") as f:
    f.write(result.stdout)
print(f"    [INFO] Disassembly -> {asm_path}")

    [INFO] Shared lib -> output/intrinsics/intrinsic_add.so
    [INFO] Disassembly -> output/intrinsics/intrinsic_add.asm
